# Validate Code Eval Pipeline: MBPP + HumanEval + LiveCodeBench\n\nTest the full execution-based evaluation pipeline with **Qwen3-0.6B** (base model).\n- **MBPP**: trainable, has ground-truth solutions + tests\n- **HumanEval**: eval-only (164 problems), function-level tests\n- **LiveCodeBench**: eval-only (~600 problems), stdin/stdout competitive programming tests

In [2]:
import sys, os
# Ensure project root is on sys.path
for p in [os.getcwd(), "/workspace/mod_gpt"]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from data.pt_dataset import (
    get_dataset, MBPPDataset, HumanEvalDataset, LiveCodeBenchDataset,
    sandbox_execute, check_code_correctness,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen3-0.6B"
print(f"Device: {DEVICE} | CWD: {os.getcwd()}")

ModuleNotFoundError: No module named 'data.pt_dataset'

## 1. Load Model & Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16
).to(DEVICE).eval()
print(f"Model loaded: {MODEL_NAME} ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")

## 2. Load All Three Datasets & Inspect Samples

In [ ]:
# --- MBPP ---
mbpp_ds = get_dataset("mbpp", split="test", tokenizer=tokenizer, max_length=1024)
print(f"MBPP test: {len(mbpp_ds)} problems")
ex = mbpp_ds.dataset[0]
print(f"  Prompt: {ex['prompt'][:150]}...")
tests = mbpp_ds.get_test_cases(0)
print(f"  Tests ({len(tests)}): {tests[0][:80]}...")

# --- HumanEval ---
he_ds = get_dataset("humaneval", split="test", tokenizer=tokenizer, max_length=1024)
print(f"\nHumanEval: {len(he_ds)} problems")
ex = he_ds.dataset[0]
print(f"  Entry point: {ex['entry_point']}")
print(f"  Prompt: {ex['prompt'][:150]}...")
tests = he_ds.get_test_cases(0)
print(f"  Tests ({len(tests)}): {tests[0][:100]}...")

# --- LiveCodeBench ---
lcb_ds = get_dataset("livecodebench", split="test", tokenizer=tokenizer, max_length=1024)
print(f"\nLiveCodeBench: {len(lcb_ds)} problems")
ex = lcb_ds.dataset[0]
print(f"  Columns: {list(ex.keys())}")
question = ex.get("question_content", ex.get("question", ""))
print(f"  Question: {str(question)[:200]}...")
tests = lcb_ds.get_test_cases(0)
print(f"  Tests ({len(tests)}): {type(tests[0]) if tests else 'none'}")

In [ ]:
## 3. Sanity Check: Ground-Truth Solutions Pass Their Own Tests\n\nVerify the sandbox works by running **reference solutions** against their tests.\n- MBPP: has `code` field with solutions\n- HumanEval: has `canonical_solution` field\n- LiveCodeBench: no reference solutions (skip)

## was: Sanity Check section header (merged into cell 6)

In [ ]:
import json

# --- MBPP: ground-truth check ---
print("=== MBPP: Ground-truth solution check ===")
for i in range(5):
    ex = mbpp_ds.dataset[i]
    code = ex["code"]
    tests = mbpp_ds.get_test_cases(i)
    result = check_code_correctness(code, tests, timeout=10)
    status = "✅" if result["passed"] else f"❌ ({result['errors'][:1]})"
    print(f"  Problem {i}: {status}  ({result['num_passed']}/{result['num_total']} tests)")

# --- HumanEval: ground-truth check ---
print("\n=== HumanEval: Ground-truth solution check ===")
for i in range(5):
    ex = he_ds.dataset[i]
    # HumanEval: prompt contains the function signature, canonical_solution is the body
    code = ex["prompt"] + ex["canonical_solution"]
    tests = he_ds.get_test_cases(i)
    result = check_code_correctness(code, tests, timeout=10)
    status = "✅" if result["passed"] else f"❌ ({result['errors'][:1]})"
    print(f"  Problem {i} ({ex['entry_point']}): {status}")

# --- LiveCodeBench: just verify test case parsing ---
print("\n=== LiveCodeBench: Test case parsing check ===")
lcb_with_tests = 0
for i in range(min(50, len(lcb_ds))):
    tests = lcb_ds.get_test_cases(i)
    if tests:
        lcb_with_tests += 1
        if lcb_with_tests <= 3:
            print(f"  Problem {i}: {len(tests)} test cases found")
print(f"  {lcb_with_tests}/50 problems have parseable test cases")

## 4. Generate Code with Qwen3-0.6B\n\nTest model generation + execution on a few problems from each benchmark.

In [ ]:
@torch.no_grad()
def generate_code(model, tokenizer, prompt_text, max_new_tokens=256):
    """Generate code from a prompt using the base HF model."""
    inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def build_prompt(dataset, idx):
    """Build the generation prompt for a given dataset and index."""
    ex = dataset.dataset[idx]
    if isinstance(dataset, MBPPDataset):
        return f"# Task: {ex['prompt'].strip()}\n# Solution:\n"
    elif isinstance(dataset, HumanEvalDataset):
        return ex["prompt"]  # already contains function sig + docstring
    else:  # LiveCodeBench
        question = ex.get("question_content", ex.get("question", "")).strip()
        starter = ex.get("starter_code", "").strip()
        if starter:
            return f"# Problem:\n{question}\n\n{starter}\n# Solution:\n"
        return f"# Problem:\n{question}\n\n# Solution:\n"


def run_demo(dataset, name, n=3, max_new_tokens=256):
    """Generate + execute on a few problems from a dataset."""
    print(f"=== {name}: Model Generation + Execution ===\n")
    tested = 0
    for i in range(min(100, len(dataset))):
        tests = dataset.get_test_cases(i)
        if not tests:
            continue
        prompt = build_prompt(dataset, i)
        generated = generate_code(model, tokenizer, prompt, max_new_tokens=max_new_tokens)
        # For HumanEval, the generated code is the function body; prepend prompt
        if isinstance(dataset, HumanEvalDataset):
            full_code = prompt + generated
        else:
            full_code = generated
        result = check_code_correctness(full_code, tests, timeout=10)
        status = "✅" if result["passed"] else "❌"
        print(f"--- Problem {i} {status} ({result['num_passed']}/{result['num_total']}) ---")
        print(f"Prompt: {prompt[:120]}...")
        print(f"Generated:\n{generated[:300]}")
        if result["errors"]:
            print(f"Error: {result['errors'][0][:200] if result['errors'][0] else 'none'}")
        print()
        tested += 1
        if tested >= n:
            break


run_demo(mbpp_ds, "MBPP", n=3, max_new_tokens=256)

In [ ]:
run_demo(he_ds, "HumanEval", n=3, max_new_tokens=256)

run_demo(lcb_ds, "LiveCodeBench", n=3, max_new_tokens=512)

In [ ]:
## 5. Batch Eval: Pass@1 on N samples\n\nRun a proper pass@1 evaluation loop on a small subset of each benchmark.

from tqdm import tqdm

@torch.no_grad()
def eval_pass_at_1(model, tokenizer, dataset, n_samples=20, max_new_tokens=256, timeout=10):
    """Compute pass@1 on a code dataset with execution-based checking."""
    passed = 0
    tested = 0
    skipped = 0
    
    for i in tqdm(range(min(n_samples * 3, len(dataset))), desc="Evaluating"):
        tests = dataset.get_test_cases(i)
        if not tests:
            skipped += 1
            continue
        if tested >= n_samples:
            break
        
        prompt = build_prompt(dataset, i)
        generated = generate_code(model, tokenizer, prompt, max_new_tokens=max_new_tokens)
        
        # For HumanEval, prepend the function signature to the generated body
        if isinstance(dataset, HumanEvalDataset):
            full_code = prompt + generated
        else:
            full_code = generated
        
        result = check_code_correctness(full_code, tests, timeout=timeout)
        if result["passed"]:
            passed += 1
        tested += 1
    
    acc = passed / max(tested, 1)
    print(f"\npass@1 = {passed}/{tested} = {acc:.1%}  (skipped {skipped} w/o tests)")
    return {"pass_at_1": acc, "passed": passed, "tested": tested, "skipped": skipped}


N = 20  # samples per benchmark

print("=" * 60)
print(f"MBPP pass@1 ({N} samples) — in-distribution")
print("=" * 60)
mbpp_result = eval_pass_at_1(model, tokenizer, mbpp_ds, n_samples=N, max_new_tokens=256)

print()
print("=" * 60)
print(f"HumanEval pass@1 ({N} samples) — OOD eval")
print("=" * 60)
he_result = eval_pass_at_1(model, tokenizer, he_ds, n_samples=N, max_new_tokens=256)

print()
print("=" * 60)
print(f"LiveCodeBench pass@1 ({N} samples) — OOD eval")
print("=" * 60)
lcb_result = eval_pass_at_1(model, tokenizer, lcb_ds, n_samples=N, max_new_tokens=512)

# --- Summary table ---
print("\n" + "=" * 60)
print("SUMMARY: Qwen3-0.6B (base, no fine-tuning)")
print("=" * 60)
print(f"{'Benchmark':<20} {'pass@1':>10} {'Passed':>8} {'Tested':>8}")
print("-" * 50)
for name, r in [("MBPP (ID)", mbpp_result), ("HumanEval (OOD)", he_result), ("LiveCodeBench (OOD)", lcb_result)]:
    print(f"{name:<20} {r['pass_at_1']:>9.1%} {r['passed']:>8} {r['tested']:>8}")
print("-" * 50)
print("ID = in-distribution (train on MBPP), OOD = out-of-distribution")